# Open-loop replay — does the policy reproduce TRAINING actions from TRAINING frames?

Decisive test for the **"robot stands still"** issue. It mirrors the RTC `policy_server`'s
exact preprocessing (`make_pre_post_processors` → `predict_action_chunk` → `postprocessor`)
but feeds frames straight from the **dataset** instead of the live robot.

**Why:** at deploy the model emits a near-constant, near-home action (std 1–3.5° vs dataset
16–45°), gripper frozen — even after resetting to an in-distribution start pose. So the
start-pose-OOD hypothesis no longer explains it. This separates the two remaining causes:

| Result | Conclusion |
|---|---|
| pred **tracks** ground-truth & **varies** across frames | model + pipeline FINE → fault is the **deploy observation pipeline** (camera images / state scaling at run time) |
| pred **~constant** / does not match GT | **model / normalization / preprocessing** is broken (despite "overfit") |

Run top-to-bottom on the **GPU server** (needs the checkpoint + dataset cache).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import get_policy_class, make_pre_post_processors

MODEL = "di-techinnova/smolvla-pouring-0.1"
DATASET = "di-techinnova/so-arm-101-pouring-0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
JOINTS = ["sh_pan", "sh_lift", "elbow", "wr_flex", "wr_roll", "grip"]
print("device:", DEVICE)

## 1. Load policy + the server's pre/post processors, and verify normalization stats are real

In [ ]:
policy = get_policy_class("smolvla").from_pretrained(MODEL).to(DEVICE).eval()
pre, post = make_pre_post_processors(
    policy.config,
    pretrained_path=MODEL,
    preprocessor_overrides={
        "device_processor": {"device": DEVICE},
        "rename_observations_processor": {"rename_map": {}},  # dataset keys already match the model
    },
    postprocessor_overrides={"device_processor": {"device": DEVICE}},
)

# Normalization buffers MUST be non-trivial (not mean=0 / std=1). If they are identity,
# the unnormalization is broken and the raw output collapses toward ~0 (a standstill).
print("--- action / state normalization buffers (first 6 dims) ---")
for name, buf in policy.named_buffers():
    low = name.lower()
    if ("action" in low or "observation.state" in low) and any(s in low for s in ("mean", "std", "min", "max")):
        print(f"  {name}: {np.round(buf.detach().cpu().numpy().ravel()[:6], 3)}")

## 2. Replay dataset frames spanning episode 0 (approach → pour)

In [ ]:
ds = LeRobotDataset(DATASET)
try:
    f0 = int(ds.episode_data_index["from"][0]); t0 = int(ds.episode_data_index["to"][0])
except Exception:
    f0, t0 = 0, 450
offsets = [0, 75, 150, 270, 350, 440]
idxs = [f0 + o for o in offsets if f0 + o < t0]
img_keys = [k for k in policy.config.input_features if k.startswith("observation.images.")]
print("model image inputs:", img_keys)
print("dataset has        :", [k for k in img_keys if k in ds[f0]])
print("replaying global idx:", idxs)

preds, gts, frames = [], [], []
for gi in idxs:
    item = ds[gi]
    obs = {"observation.state": item["observation.state"].unsqueeze(0)}
    for k in img_keys:
        if k in item:                       # camera3 absent in dataset -> dropped by prepare_images
            obs[k] = item[k].unsqueeze(0)
    obs["task"] = item.get("task", "Pour from orange cup into blue cup.")

    policy.reset()
    processed = pre(obs)
    with torch.no_grad():
        chunk = policy.predict_action_chunk(processed)     # (B, chunk, dim), normalized
    if chunk.ndim != 3:
        chunk = chunk.unsqueeze(0)
    pred = post(chunk[:, 0, :]).detach().cpu().numpy().ravel()[:6]   # unnormalized first action
    gt = item["action"].detach().cpu().numpy().ravel()[:6]
    fi = int(item["frame_index"].item()) if "frame_index" in item else gi - f0
    preds.append(pred); gts.append(gt); frames.append(fi)
    print(f"frame {fi:3d} | pred {np.round(pred,1)} | gt {np.round(gt,1)} | |err| {np.round(np.abs(pred-gt),1)}")

preds = np.stack(preds); gts = np.stack(gts); frames = np.array(frames)

## 3. Verdict

In [ ]:
mae = float(np.abs(preds - gts).mean())
pred_var = preds.std(0); gt_var = gts.std(0)
print(f"mean |pred - GT|              : {mae:.2f} deg")
print(f"pred variation across frames  : {np.round(pred_var,1)}")
print(f"GT   variation across frames  : {np.round(gt_var,1)}")
print()
reproduces = mae < 8 and pred_var.mean() > 0.4 * gt_var.mean()
if reproduces:
    print("=> MODEL REPRODUCES TRAINING DATA.")
    print("   Fault is the DEPLOY observation pipeline: compare the deploy camera images")
    print("   and state scaling against the dataset (next notebook / step).")
else:
    print("=> MODEL DOES NOT REPRODUCE (output near-constant or off).")
    print("   Fault is the MODEL or normalization/preprocessing:")
    print("   - check the buffers in section 1 are non-identity")
    print("   - re-check the checkpoint (training steps / loss) and from_pretrained path")

In [ ]:
# pred vs GT per joint across the sampled frames
fig, axs = plt.subplots(2, 3, figsize=(13, 6))
for j, ax in enumerate(axs.ravel()):
    ax.plot(frames, gts[:, j], "o-", label="ground-truth", color="tab:green")
    ax.plot(frames, preds[:, j], "x--", label="pred", color="tab:red")
    ax.set_title(JOINTS[j]); ax.set_xlabel("frame"); ax.grid(alpha=.3)
    if j == 0:
        ax.legend()
fig.suptitle("Open-loop: predicted vs ground-truth action per joint (episode 0)")
plt.tight_layout(); plt.show()

## 4. How to read this

- **Lines overlap & both move** (pred follows GT from approach to pour) → the model and the
  whole inference pipeline are correct. The standstill is then a **deploy-time observation**
  problem: the live camera images or the `observation.state` the robot reports differ from
  what the model saw in training (color/space/resolution/crop, camera assignment, or a state
  scale/units mismatch). Next: dump a live deploy observation and diff it against a dataset frame.

- **Red (pred) is flat / far from green** → the model itself isn't conditioning on inputs.
  Re-check: the normalization buffers in §1 (identity ⇒ broken), the checkpoint actually
  converged (training loss), and that `from_pretrained` loaded the intended weights. An
  undertrained or mis-saved checkpoint produces exactly this near-constant output.